# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint
### Part-A: Finetuning a Seq2Seq (T5) Model for Summarization

## Reference notebook not for submission

> **NOTE that** this Assignment is in 2 parts:
> - Part-A: Finetuning a Seq2Seq (T5) Model for Summarization
> - Part-B: PEFT for Dialogue Summary
>
>Only Part-B needs to be submitted for grading.

## Learning Objectives

At the end of the experiment, you will be able to:

* fine tune a T5 model, `facebook/bart-large-cnn`, on the SAMSum dataset for summerization
* push the finetune model to HuggingFace model hub
* load the finetuned model from hub for inference

## Dataset Description

The **[SAMSum](https://huggingface.co/datasets/samsum) dataset** contains about 16k messenger-like conversations with summaries. Conversations were created and written down by linguists fluent in English. Linguists were asked to create conversations similar to those they write on a daily basis, reflecting the proportion of topics of their real-life messenger convesations. The style and register are diversified - conversations could be informal, semi-formal or formal, they may contain slang words, emoticons and typos. Then, **the conversations were annotated with summaries**. It was assumed that summaries should be a concise brief of what people talked about in the conversation in third person. The SAMSum dataset was prepared by Samsung R&D Institute Poland and is distributed for research purposes.

Data Splits:
- train: 14732
- val: 818
- test: 819

Data Fields:

- ***dialogue***: text of dialogue
- ***summary***: human written summary of the dialogue
- ***id***: unique id of an example

<br>

**Example:**

\{
> '**id**': '13818513',

>'**summary**': 'Amanda baked cookies and will bring Jerry some tomorrow.',

>'**dialogue**': "Amanda: I baked cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

\}

## Information

**Summarization** creates a shorter version of a document or an article that captures all the important information. Along with translation, it is another example of a task that can be formulated as a sequence-to-sequence task.

**BART** is a transformer encoder-decoder (seq2seq) model with a bidirectional (BERT-like) encoder and an autoregressive (GPT-like) decoder.

BART is pre-trained by
1. corrupting text with an arbitrary noising function, and
2. learning a model to reconstruct the original text.

BART is particularly effective when fine-tuned for text generation (e.g. summarization, translation) but also works well for comprehension tasks (e.g. text classification, question answering). This particular checkpoint, `facebook/bart-large-cnn`, has been fine-tuned on CNN Daily Mail dataset, a large collection of text-summary pairs.

To know more about BART `facebook/bart-large-cnn`, refer to its Model card [here](https://huggingface.co/facebook/bart-large-cnn).

### Install required dependencies

In [1]:
#!pip install "transformers==4.27.2" "datasets==2.9.0" "accelerate==0.17.1" "evaluate==0.4.0" "bitsandbytes==0.40.2" loralib --upgrade --quiet
# install additional dependencies needed for training

In [2]:
# HuggingFace transformers and datasets
!pip -q install transformers datasets

In [3]:
# 'Accelerate' is the backend for the PyTorch side
#  It enables the PyTorch code to be run across any distributed configuration
!pip -q install accelerate -U


# To install both 'transformer' and 'accelerate' in one go
# !pip install transformers[torch]

In [4]:
# A dependecy required for loading SAMSum dataset
!pip -q install py7zr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.7/142.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 429.9/429.9 kB 19.0 MB/s eta 0:00:00


In [5]:
!pip -q install transformers

### Import required packages

In [6]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import TrainingArguments, Trainer

import warnings
warnings.filterwarnings('ignore')

### **Load Model & Tokenizer**

* **Load the model and tokenizer from HF Model Hub for finetuning**

    - In many cases, the architecture you want to use can be guessed from the name or the path of the pretrained model you are supplying to the `from_pretrained()` method. **AutoClasses** can be used to automatically retrieve the relevant model given the name/path to the pretrained weights/config/vocabulary.

    - Instantiating one of `AutoConfig`, `AutoModel`, and `AutoTokenizer` will directly create a class of the relevant architecture.

    - `AutoModelForSeq2SeqLM` instantiates one of the model classes of the library (with a sequence-to-sequence language modeling head) from a configuration.

    - Full path of the model repo needs to be specified i.e. ***''USER-NAME/REPO-NAME''*** while calling `from_pretrained()` method.

In [7]:
# Load model from HF Model Hub

"""
BART HAS 400M PARAMS: https://github.com/facebookresearch/fairseq/tree/main/examples/bart
Look into Model card - 400 Million parameters
"""

checkpoint = "facebook/bart-large-cnn"                # username/repo-name

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

### **Load Dataset**

In [9]:
# Load SAMSum dataset
dataset = load_dataset("knkarthick/samsum")
dataset

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

### **Testing the pre-trained model**

#### Observing the data

In [10]:
sample = dataset['test'][0]['dialogue']
label = dataset['test'][0]['summary']
print(sample,'\n','--------------')
print(label)

Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye 
 --------------
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


#### Prompt Preparation

In [28]:
def generate_summary(input, llm):
    """Prepare prompt  -->  tokenize -->  generate output using LLM  -->  detokenize output"""

    input_prompt = f"""
                    Summarize the following conversation.

                    {input}

                    Summary:
                    """

    input_ids = tokenizer(input_prompt, return_tensors='pt')
    tokenized_output = llm.generate(input_ids=input_ids['input_ids'], min_length=30, max_length=200, generation_config=generation_config)
    output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)

    return output

#### Getting the output

In [12]:
output = generate_summary(sample, llm=model)
print("Sample")
print(sample)
print("-------------------")
print("Model Generated Summary:")
print(output)
print("Correct Summary:")
print(label)

Sample
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
-------------------
Model Generated Summary:
Hannah asks Amanda for Betty's number. Amanda tries to find the number but can't find it. She asks Hannah to text Larry. Hannah says she'd rather text him.
Correct Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


### **Prepare Dataset**

In [13]:
# Define function to prepare dataset

def tokenize_inputs(example):

    start_prompt = "Summarize the following conversation.\n\n"
    end_prompt = "\n\nSummary: "
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example['dialogue']]
    example['input_ids'] = tokenizer(prompt, padding='max_length', truncation=True, max_length=512, return_tensors='pt').input_ids             # 'pt' for pytorch tensor
    example['labels'] = tokenizer(example['summary'], padding='max_length', truncation=True, max_length=512, return_tensors='pt').input_ids

    return example

In the below code, we are using `batched=True` to use Fast tokenizer implementation.

**Slow** tokenizers are those written in Python inside the HF Transformers library, while the **fast** versions are the ones provided by HF Tokenizers, which are written in Rust.

To know more about 'slow' and 'fast' tokenizers, refer [here](https://huggingface.co/learn/nlp-course/chapter6/3?fw=pt)

In [14]:
# Prepare dataset
tokenizer.pad_token = tokenizer.eos_token
tokenized_datasets = dataset.map(tokenize_inputs, batched=True)       # using batched=True for Fast tokenizer implementation

# Remove columns/keys that are not needed further
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'dialogue', 'summary'])

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [15]:
# Shortening the data: Just picking row index divisible by 100
# For learning purpose! It will reduce the compute resource requirement and training time

tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 100 == 0, with_indices=True)

Filter:   0%|          | 0/14731 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818 [00:00<?, ? examples/s]

Filter:   0%|          | 0/819 [00:00<?, ? examples/s]

In [16]:
print(tokenized_datasets['train'].shape)
print(tokenized_datasets['validation'].shape)
print(tokenized_datasets['test'].shape)

(148, 2)
(9, 2)
(9, 2)


In [17]:
tokenized_datasets['train'][0].keys()

dict_keys(['input_ids', 'labels'])

### **Define Training Arguments and Trainer object**

**To upload the finetuned model on HF Model Hub, first you need to create a HuggingFace Account and Create a new model repository and Access Tokens with read/write permission**
* [Sign up](https://huggingface.co/join) for a Hugging Face account
    
        * Follow the below steps to create reposotory
    
            - By going through your icon on huggingface you will find new model.
            - Create your Model name, with License as ( MIT/mit ), keep it public and create model.
            - You can access your folder location from the browser URL : https://huggingface.co/[YOUR-USER-NAME]/[YOUR-MODEL-REPO-NAME]
            - With your user name and model repo name in training arguments uncomment and rename them `"sumanthk/PEFT_expo"`



In [19]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bart-cnn-samsum-finetuned",        # local directory
    hub_model_id="sumanthk/PEFT_expo",      # identifier on the Hub for directly pushing to HFhub model
    learning_rate=1e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    auto_find_batch_size=True,
    eval_strategy='epoch',
    logging_steps=10,
)

In [20]:
from transformers import GenerationConfig

In [21]:
# Configure generation settings

generation_config = GenerationConfig(
    max_length=142,  # Maximum length of generated sequences
    min_length=56,  # Minimum length of generated sequences
    early_stopping=True,  # Stop generation early if all beams reach an EOS token
    num_beams=4,  # Number of beams for beam search
    length_penalty=2.0,  # Penalty for longer sequences
    no_repeat_ngram_size=3,  # Prevent repeating n-grams
    forced_bos_token_id=0,  # Force the beginning of sequence token
    forced_eos_token_id=2,  # Force the end of sequence token
)

model.generation_config.decoder_start_token_id = tokenizer.cls_token_id

In [22]:
trainer = Trainer(
    model=model,           # model to be finetuned
    tokenizer=tokenizer,       # tokenizer to use
    args=training_args,        # training arguments such as epochs, learning_rate, etc
    train_dataset=tokenized_datasets['train'],         # training data to use
    eval_dataset=tokenized_datasets['validation'],     # validation data to use
)

In [23]:
# Disabling Weights and Biases logging
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
#9960b5b271d65c92b2a9332316599af8ba6f7c06

In [24]:
# Training
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aiml-support (hiteshteam1) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,2.222900,0.803663
2,0.225200,0.604783


TrainOutput(global_step=38, training_loss=0.7561925994722467, metrics={'train_runtime': 623.76, 'train_samples_per_second': 0.475, 'train_steps_per_second': 0.061, 'total_flos': 320731481112576.0, 'train_loss': 0.7561925994722467, 'epoch': 2.0})

### **Save the model on Local system**

In [25]:
ver = 1
output_directory="./bart-cnn-samsum-finetuned"
model_path = os.path.join(output_directory, f"tuned_model_{ver}" )

# Save finetuned model
trainer.save_model(model_path)

# Save associated tokenizer
tokenizer.save_pretrained(model_path)

print(f"\nSaved at path: {model_path}")


Saved at path: ./bart-cnn-samsum-finetuned/tuned_model_1


### **Load the model from Local system and test**

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model4local = AutoModelForSeq2SeqLM.from_pretrained(model_path)

In [29]:
output = generate_summary(sample, llm = model4local)

print("Sample")
print(sample)
print("-------------------")
print("Summary:")
print(output)
print("Ground Truth Summary:")
print(label)

`generation_config` default values have been modified to match model-specific defaults: {'pad_token_id': 2, 'bos_token_id': 0, 'eos_token_id': 2, 'decoder_start_token_id': 2}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
-------------------
Summary:
Hannah asks Amanda for Betty's number. Amanda can't find it. She asks Hannah to text Larry. Hannah says she'll text him back.
Ground Truth Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


### **Push your model to Hugging Face Model Hub**

To upload the finetuned model on HF Model Hub, please follow below steps:

**Steps to push your fine-tuned model to HuggingFace Model Hub:**

1. Go to already signed up Hugging Face account

2. Create an access token for your account and save it

    To create an access token:
    
        - Go to your `Settings`, then click on the `Access Tokens` tab. Click on the `New token` button to create a new User Access Token.
        - Select a role(`write`) and a name for your token
        - Click Generate a token

    To know more about Access Tokens, refer [here](https://huggingface.co/docs/hub/security-tokens).


3. Once you have your User Access Token, run the following command to authenticate your identity to the Hub.
        - `notebook_login()`
        - Paste your Access token when prompted
    
    For more details on login, refer [here](https://huggingface.co/docs/huggingface_hub/quick-start#login).

In [32]:
from huggingface_hub import notebook_login

notebook_login()

4. Push your fine-tuned model and tokenizer to Model Hub
        - Use `push_to_hub()` method of your model and tokenizer both, to push them on hub

In [33]:
trainer.push_to_hub()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...model_1/model.safetensors:   0%|          |  174kB / 1.63GB            

  ...netuned/model.safetensors:   0%|          |  174kB / 1.63GB            

  ...163269.cc8e6c9311cc.923.0:   9%|9         |   696B / 7.54kB            

  ...netuned/training_args.bin:   9%|9         |   545B / 5.91kB            

  ...model_1/training_args.bin:   9%|9         |   545B / 5.91kB            

CommitInfo(commit_url='https://huggingface.co/sumanthk/PEFT_expo/commit/bb3f58c53da02ad92d6cdc72e9b561c4f4a0a9e4', commit_message='End of training', commit_description='', oid='bb3f58c53da02ad92d6cdc72e9b561c4f4a0a9e4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sumanthk/PEFT_expo', endpoint='https://huggingface.co', repo_type='model', repo_id='sumanthk/PEFT_expo'), pr_revision=None, pr_num=None)

### **Test your finetuned model downloaded from HF Model Hub**

- Specify user name and your repository where the model and tokenizer will be loaded from.
    

In [34]:
username = "sumanthk"      # change it to your HuggingFace username

checkpoint = username + '/PEFT_expo'  # change it to your Repo name

loaded_model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

In [35]:
output = generate_summary(sample, llm=loaded_model)

print("Sample")
print(sample)
print("-------------------")
print("Summary:")
print(output)
print("Ground Truth Summary:")
print(label)

`generation_config` default values have been modified to match model-specific defaults: {'pad_token_id': 2, 'bos_token_id': 0, 'eos_token_id': [2], 'decoder_start_token_id': 0}. If this is not desired, please set these values explicitly.


Sample
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
-------------------
Summary:
Hannah asks Amanda for Betty's number. Amanda says she doesn't know Betty but Larry called her last time they were at the park together. Hannah wants to text Larry.
Ground Truth Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


### References

- [Summarization](https://huggingface.co/docs/transformers/tasks/summarization)